# Set up

In [ ]:
!pip install --upgrade transformers bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 57.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 16.3 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


In [ ]:
from datasets import load_dataset

ds = load_dataset("dgslibisey/MuSiQue", split="train[:300]")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


musique_ans_v1.0_train.jsonl:   0%|          | 0.00/241M [00:00<?, ?B/s]

musique_ans_v1.0_dev.jsonl:   0%|          | 0.00/30.4M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/19938 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2417 [00:00<?, ? examples/s]

In [ ]:
ds.shape

(300, 7)

In [ ]:
ds[0]

{'id': '2hop__482757_12019',
 'paragraphs': [{'idx': 0,
   'title': 'Pakistan Super League',
   'paragraph_text': 'Pakistan Super League (Urdu: پاکستان سپر لیگ \u202c \u200e; PSL) is a Twenty20 cricket league, founded in Lahore on 9 September 2015 with five teams and now comprises six teams. Instead of operating as an association of independently owned teams, the league is a single entity in which each franchise is owned and controlled by investors.',
   'is_supporting': False},
  {'idx': 1,
   'title': 'Serena Wilson',
   'paragraph_text': 'Serena Wilson (August 8, 1933 – June 17, 2007), often known just as "Serena", was a well-known dancer, choreographer, and teacher who helped popularize belly dance in the United States. Serena\'s work also helped legitimize the dance form and helped it to be perceived as more than burlesque or stripping. Serena danced in clubs in her younger years, opened her own studio, hosted her own television show, founded her own dance troupe, and was the auth

In [ ]:
import pandas as pd

In [ ]:
def simplify_musique_dataset(raw_dataset):
    simplified_data = []

    for item in raw_dataset:
        # On extrait et numérote proprement les paragraphes
        # idx 0, 1, 2... pour que le LLM puisse les citer facilement
        contexts = [
            {"id": p['idx'], "text": p['paragraph_text']}
            for p in item['paragraphs']
        ]

        # On récupère les IDs des paragraphes "supporting"
        gold_ids = [p['idx'] for p in item['paragraphs'] if p['is_supporting']]

        simplified_data.append({
            "id": item['id'],
            "question": item['question'],
            "contexts": contexts,
            "gold_ids": gold_ids,
            "answer": item['answer']
        })
    df = pd.DataFrame(simplified_data)
    return df



In [ ]:
# Utilisation
ds_rag = simplify_musique_dataset(ds)

In [ ]:
ds_rag.head()

,id,question,contexts,gold_ids,answer
0,2hop__482757_12019,When was the institute that owned The Collegia...,"[{'id': 0, 'text': 'Pakistan Super League (Urd...","[5, 9]",1960
1,2hop__129292_160863,What year saw the creation of the region where...,"[{'id': 0, 'text': '""The Spy in Black"" was fil...","[6, 10]",1994
2,2hop__679261_120616,When was the abolishment of the studio that di...,"[{'id': 0, 'text': 'PolyGram Filmed Entertainm...","[0, 17]",1999
3,2hop__813857_127131,When was the publisher of Crux launched?,"[{'id': 0, 'text': 'PopSister was a Japanese m...","[12, 17]",1998
4,2hop__144408_215084,Jan Šindel's was born in what country?,"[{'id': 0, 'text': 'Jan Anton van der Baren (v...","[3, 11]",Czech Republic


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def load_llama_model(model_id="meta-llama/Llama-3.1-8B-Instruct"):
    # 1. Configuration de la quantification 4-bit
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16
    )

    # 2. Chargement du Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token # Indispensable pour Llama

    # 3. Chargement du Modèle
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )

    return model, tokenizer


In [ ]:
model, tokenizer = load_llama_model()

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

## tester avec qwen

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

def load_qwen_model(model_id="Qwen/Qwen2.5-7B-Instruct"):
    # 1. Configuration de la quantification 4-bit
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )

    # 2. Chargement du Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_id)

    # 3. Chargement du Modèle
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True
    )

    return model, tokenizer

# model, tokenizer = load_qwen_model()

In [ ]:
model, tokenizer = load_qwen_model()

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

## reste

In [ ]:
def prepare_input(row, prompt_template):
    # Formate les paragraphes avec leurs IDs
    context_text = "\n".join([f"ID: {p['id']} | Paragraph: {p['text']}" for p in row['contexts']])

    # Injection dans le template
    full_prompt = prompt_template.format(
        context_text=context_text,
        question=row['question']
    )
    return full_prompt

In [ ]:
import re

def process_output_with_tags(raw_output):
    # 1. On cherche ce qu'il y a entre <answer> et </answer>
    answer_match = re.search(r"<answer>(.*?)</answer>", raw_output, re.DOTALL)
    answer = answer_match.group(1).strip() if answer_match else "Format Error (Answer)"

    # 2. On cherche ce qu'il y a entre <chunks_id> et </chunks_id>
    chunks_match = re.search(r"<chunks_id>(.*?)</chunks_id>", raw_output, re.DOTALL)

    predicted_ids = []
    if chunks_match:
        # On extrait tous les nombres présents dans la balise
        predicted_ids = [int(n) for n in re.findall(r'\d+', chunks_match.group(1))]

    return answer, predicted_ids

In [ ]:
import gc
import torch

def clean_memory():
    # 1. Force le ramasse-miettes de Python à passer
    gc.collect()
    # 2. Vide le cache inutilisé de la mémoire GPU
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        # Optionnel : synchronise pour s'assurer que tout est propre
        torch.cuda.synchronize()

# Baseline

In [ ]:
PROMPT_BASELINE = """You are a precise QA assistant. Your task is to answer the question using ONLY the provided paragraphs.

### INSTRUCTIONS:
1. **Evidence-Based Only**: Provide an accurate answer based EXCLUSIVELY on the provided text.
2. **Handling Absence**: If information is missing, write "Insufficient information" inside the tags.
3. **Mandatory Attribution**: You MUST identify the IDs of the paragraphs used.
4. **Strict Output Format**: You must wrap your response and the IDs using the specific XML-style tags provided below.

### FORMAT:
You must STRICTLY follow this format :
<answer>[Your concise answer here]</answer>
<chunks_id>[List only the numerical IDs separated by commas]</chunks_id>

### CONTEXT:
{context_text}

### QUESTION:
{question}
"""

In [ ]:
model, tokenizer = load_llama_model()

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

In [ ]:
results_metrics = []

for index, row in ds_rag.head(20).iterrows():

    full_prompt = prepare_input(row, PROMPT_BASELINE)

    messages = [{"role": "user", "content": full_prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=350,
            do_sample=False
        )

    raw_output = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    predicted_answer, predicted_ids = process_output_with_tags(raw_output)

    gold_set = set(row['gold_ids'])
    pred_set = set(predicted_ids)

    correct_ids = pred_set.intersection(gold_set)

    p = len(correct_ids) / len(pred_set) if len(pred_set) > 0 else 0
    r = len(correct_ids) / len(gold_set) if len(gold_set) > 0 else 0
    em = 1 if pred_set == gold_set else 0

    results_metrics.append({"p": p, "r": r, "em": em})

    print(f"--- SAMPLE {index+1} ---")
    print(f"Gold IDs: {row['gold_ids']}")
    print(f"Pred IDs: {predicted_ids}")
    print(f"Verdict: {'✅' if em else '❌'} (P: {p:.2f}, R: {r:.2f})")
    print("-" * 30)

    del inputs, outputs
    clean_memory()

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 1 ---
Gold IDs: [5, 9]
Pred IDs: [9]
Verdict: ❌ (P: 1.00, R: 0.50)
------------------------------
--- SAMPLE 2 ---
Gold IDs: [6, 10]
Pred IDs: [10]
Verdict: ❌ (P: 1.00, R: 0.50)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 3 ---
Gold IDs: [0, 17]
Pred IDs: [0]
Verdict: ❌ (P: 1.00, R: 0.50)
------------------------------
--- SAMPLE 4 ---
Gold IDs: [12, 17]
Pred IDs: [17]
Verdict: ❌ (P: 1.00, R: 0.50)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

--- SAMPLE 18 ---
Gold IDs: [4, 13]
Pred IDs: [13, 8]
Verdict: ❌ (P: 0.50, R: 0.50)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 19 ---
Gold IDs: [8, 14]
Pred IDs: [8]
Verdict: ❌ (P: 1.00, R: 0.50)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 20 ---
Gold IDs: [3, 9]
Pred IDs: [9]
Verdict: ❌ (P: 1.00, R: 0.50)
------------------------------


In [ ]:
total = len(results_metrics)
print("\n" + "="*40)
print(f"BILAN FINAL NLI-SCAN ({total} SAMPLES)")
print("="*40)
print(f"Précision moyenne : {sum(m['p'] for m in results_metrics)/total:.2%}")
print(f"Rappel moyen    : {sum(m['r'] for m in results_metrics)/total:.2%}")
print(f"Exact Match Total : {sum(m['em'] for m in results_metrics)/total:.2%}")


BILAN FINAL NLI-SCAN (20 SAMPLES)
Précision moyenne : 78.00%
Rappel moyen    : 47.50%
Exact Match Total : 5.00%


## prompt modifié

In [ ]:
PROMPT_BASELINE_modifie = """You are a precise QA assistant. Your task is to answer the question using ONLY the provided paragraphs.

### INSTRUCTIONS:
1. **Evidence-Based Only**: Provide an accurate answer based EXCLUSIVELY on the provided text.
2. **Handling Absence**: If information is missing, write "Insufficient information" inside the tags.
3. **Mandatory Attribution**: You MUST identify the IDs of the paragraphs used. YOU MUST identify ALL paragraphs that play a role in the reasoning chain. Include paragraphs that identify intermediate entities. Even if a paragraph only provides an indirect link or a necessary definition, it MUST be included.
4. **Strict Output Format**: You must wrap your response and the IDs using the specific XML-style tags provided below.

### FORMAT:
You must STRICTLY follow this format :
<answer>[Your concise answer here]</answer>
<chunks_id>[List only the numerical IDs separated by commas]</chunks_id>

### CONTEXT:
{context_text}

### QUESTION:
{question}
"""

In [ ]:
results_metrics = []

for index, row in ds_rag.head(20).iterrows():

    full_prompt = prepare_input(row, PROMPT_BASELINE_modifie)

    messages = [{"role": "user", "content": full_prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=350,
            do_sample=False
        )

    raw_output = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    predicted_answer, predicted_ids = process_output_with_tags(raw_output)

    gold_set = set(row['gold_ids'])
    pred_set = set(predicted_ids)

    correct_ids = pred_set.intersection(gold_set)

    p = len(correct_ids) / len(pred_set) if len(pred_set) > 0 else 0
    r = len(correct_ids) / len(gold_set) if len(gold_set) > 0 else 0
    em = 1 if pred_set == gold_set else 0

    results_metrics.append({"p": p, "r": r, "em": em})

    print(f"--- SAMPLE {index+1} ---")
    print(f"Gold IDs: {row['gold_ids']}")
    print(f"Pred IDs: {predicted_ids}")
    print(f"Verdict: {'✅' if em else '❌'} (P: {p:.2f}, R: {r:.2f})")
    print("-" * 30)

    del inputs, outputs
    clean_memory()

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 1 ---
Gold IDs: [5, 9]
Pred IDs: [9]
Verdict: ❌ (P: 1.00, R: 0.50)
------------------------------
--- SAMPLE 2 ---
Gold IDs: [6, 10]
Pred IDs: [10]
Verdict: ❌ (P: 1.00, R: 0.50)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 3 ---
Gold IDs: [0, 17]
Pred IDs: [0, 17]
Verdict: ✅ (P: 1.00, R: 1.00)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

--- SAMPLE 18 ---
Gold IDs: [4, 13]
Pred IDs: [8, 13]
Verdict: ❌ (P: 0.50, R: 0.50)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 19 ---
Gold IDs: [8, 14]
Pred IDs: [8]
Verdict: ❌ (P: 1.00, R: 0.50)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 20 ---
Gold IDs: [3, 9]
Pred IDs: [9]
Verdict: ❌ (P: 1.00, R: 0.50)
------------------------------


In [ ]:
total = len(results_metrics)
print("\n" + "="*40)
print(f"BILAN FINAL baseline prompt modifie ({total} SAMPLES)")
print("="*40)
print(f"Précision moyenne : {sum(m['p'] for m in results_metrics)/total:.2%}")
print(f"Rappel moyen    : {sum(m['r'] for m in results_metrics)/total:.2%}")
print(f"Exact Match Total : {sum(m['em'] for m in results_metrics)/total:.2%}")


BILAN FINAL baseline prompt modifie (20 SAMPLES)
Précision moyenne : 76.33%
Rappel moyen    : 52.50%
Exact Match Total : 10.00%


# prompting  : on demande d abord la liste des chunks pour répondre

In [ ]:
PROMPT_SELECTION_FIRST = """You are a precise QA assistant. Your task is to answer the question using ONLY the provided paragraphs.

### INSTRUCTIONS:
1. **Step 1 - Evidence Selection**: Identify the IDs of the paragraphs that contain the necessary information.
2. **Step 2 - Answer Generation**: Write a concise answer based EXCLUSIVELY on the paragraphs you selected in Step 1.
3. **Strict Formatting**: You must output the paragraph IDs first, then the answer, using the tags provided below.

### FORMAT:
<chunks_id>[Numerical IDs only, e.g., 5, 9]</chunks_id>
<answer>[Your concise answer here]</answer>

### CONTEXT:
{context_text}

### QUESTION:
{question}
"""

In [ ]:
results_prompt_before = []

for index, row in ds_rag.head(20).iterrows():

    full_prompt = prepare_input(row, PROMPT_SELECTION_FIRST)

    messages = [{"role": "user", "content": full_prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=350,
            do_sample=False
        )

    raw_output = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    predicted_answer, predicted_ids = process_output_with_tags(raw_output)

    gold_set = set(row['gold_ids'])
    pred_set = set(predicted_ids)

    correct_ids = pred_set.intersection(gold_set)

    p = len(correct_ids) / len(pred_set) if len(pred_set) > 0 else 0
    r = len(correct_ids) / len(gold_set) if len(gold_set) > 0 else 0
    em = 1 if pred_set == gold_set else 0

    results_prompt_before.append({"p": p, "r": r, "em": em})

    print(f"--- SAMPLE {index+1} ---")
    print(f"Gold IDs: {row['gold_ids']}")
    print(f"Pred IDs: {predicted_ids}")
    print(f"Verdict: {'✅' if em else '❌'} (P: {p:.2f}, R: {r:.2f})")
    print("-" * 30)

    del inputs, outputs
    clean_memory()




[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 1 ---
Gold IDs: [5, 9]
Pred IDs: [5, 6]
Verdict: ❌ (P: 0.50, R: 0.50)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 2 ---
Gold IDs: [6, 10]
Pred IDs: [10]
Verdict: ❌ (P: 1.00, R: 0.50)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

--- SAMPLE 17 ---
Gold IDs: [15, 17]
Pred IDs: [15]
Verdict: ❌ (P: 1.00, R: 0.50)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 18 ---
Gold IDs: [4, 13]
Pred IDs: [13]
Verdict: ❌ (P: 1.00, R: 0.50)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 19 ---
Gold IDs: [8, 14]
Pred IDs: [14]
Verdict: ❌ (P: 1.00, R: 0.50)
------------------------------
--- SAMPLE 20 ---
Gold IDs: [3, 9]
Pred IDs: [9]
Verdict: ❌ (P: 1.00, R: 0.50)
------------------------------


In [ ]:
total = len(results_prompt_before)
print("\n" + "="*40)
print(f"BILAN FINAL NLI-SCAN ({total} SAMPLES)")
print("="*40)
print(f"Précision moyenne : {sum(m['p'] for m in results_prompt_before)/total:.2%}")
print(f"Rappel moyen    : {sum(m['r'] for m in results_prompt_before)/total:.2%}")
print(f"Exact Match Total : {sum(m['em'] for m in results_prompt_before)/total:.2%}")


BILAN FINAL NLI-SCAN (20 SAMPLES)
Précision moyenne : 85.00%
Rappel moyen    : 47.50%
Exact Match Total : 5.00%


## prompt legerement modifié

In [ ]:
PROMPT_SELECTION_FIRST = """You are a precise QA assistant. Your task is to answer the question using ONLY the provided paragraphs.

### INSTRUCTIONS:
1. **Step 1 - Read carefully**: First, read carefully and with attention the question and all the paragraphs, before any generation
2. **Step 2 - Evidence Selection**: Identify the IDs of the paragraphs that contain the necessary information. YOU MUST identify ALL paragraphs that play a role in the reasoning chain. Include paragraphs that identify intermediate entities. Even if a paragraph only provides an indirect link or a necessary definition, it MUST be included.
3. **Step 3 - Answer Generation**: Write a concise answer based EXCLUSIVELY on the paragraphs you selected in Step 2.
4. **Strict Formatting**: You must output the paragraph IDs first, then the answer, using the tags provided below.

### FORMAT:
<chunks_id>[Numerical IDs only, e.g., 5, 9]</chunks_id>
<answer>[Your concise answer here]</answer>

### CONTEXT:
{context_text}

### QUESTION:
{question}
"""

In [ ]:
results_prompt_before = []

for index, row in ds_rag.head(20).iterrows():

    full_prompt = prepare_input(row, PROMPT_SELECTION_FIRST)

    messages = [{"role": "user", "content": full_prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=350,
            do_sample=False
        )

    raw_output = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    )

    predicted_answer, predicted_ids = process_output_with_tags(raw_output)

    gold_set = set(row['gold_ids'])
    pred_set = set(predicted_ids)

    correct_ids = pred_set.intersection(gold_set)

    p = len(correct_ids) / len(pred_set) if len(pred_set) > 0 else 0
    r = len(correct_ids) / len(gold_set) if len(gold_set) > 0 else 0
    em = 1 if pred_set == gold_set else 0

    results_prompt_before.append({"p": p, "r": r, "em": em})

    print(f"--- SAMPLE {index+1} ---")
    print(f"Gold IDs: {row['gold_ids']}")
    print(f"Pred IDs: {predicted_ids}")
    print(f"Verdict: {'✅' if em else '❌'} (P: {p:.2f}, R: {r:.2f})")
    print("-" * 30)

    del inputs, outputs
    clean_memory()




[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 1 ---
Gold IDs: [5, 9]
Pred IDs: [9, 6]
Verdict: ❌ (P: 0.50, R: 0.50)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 2 ---
Gold IDs: [6, 10]
Pred IDs: [10]
Verdict: ❌ (P: 1.00, R: 0.50)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

--- SAMPLE 18 ---
Gold IDs: [4, 13]
Pred IDs: [13]
Verdict: ❌ (P: 1.00, R: 0.50)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 19 ---
Gold IDs: [8, 14]
Pred IDs: [8, 14]
Verdict: ✅ (P: 1.00, R: 1.00)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 20 ---
Gold IDs: [3, 9]
Pred IDs: [9]
Verdict: ❌ (P: 1.00, R: 0.50)
------------------------------


In [ ]:
total = len(results_prompt_before)
print("\n" + "="*40)
print(f"BILAN FINALprompting before ({total} SAMPLES)")
print("="*40)
print(f"Précision moyenne : {sum(m['p'] for m in results_prompt_before)/total:.2%}")
print(f"Rappel moyen    : {sum(m['r'] for m in results_prompt_before)/total:.2%}")
print(f"Exact Match Total : {sum(m['em'] for m in results_prompt_before)/total:.2%}")


BILAN FINALprompting before (20 SAMPLES)
Précision moyenne : 89.17%
Rappel moyen    : 60.00%
Exact Match Total : 20.00%


# NLI

In [ ]:
from transformers import pipeline

# 1. Mise à jour du modèle
nli_judge = pipeline(
    "text-classification",
    model="potsawee/deberta-v3-large-mnli",
    device=0
)



config.json:   0%|          | 0.00/883 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/1.74G [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/400 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/23.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/173 [00:00<?, ?B/s]

In [ ]:
def scan_context_with_nli(answer, context_list, threshold=0.5):
    """
    answer: la phrase générée par Qwen
    context_list: la liste des chunks (row['contexts'])
    threshold: confiance minimale pour accepter l'attribution
    """
    predicted_ids = []

    for chunk in context_list:
        # 2. Utilisation de text et text_pair pour laisser le pipeline gérer les séparateurs
        result = nli_judge({"text": chunk['text'], "text_pair": answer})[0]

        # 3. .upper() ou .lower() pour éviter les erreurs de casse (ex: 'entailment' vs 'ENTAILMENT')
        if result['label'].upper() == 'ENTAILMENT' and result['score'] >= threshold:
            predicted_ids.append(chunk['id'])

    return predicted_ids

In [ ]:
import re

def scan_context_with_nli(answer, context_list, threshold=0.2):
    predicted_ids = []

    # 1. Nettoyer les balises <answer> générées par le LLM
    clean_answer = re.sub(r'</?answer>', '', answer).strip()

    # On prépare la liste de paires
    pairs = [{"text": chunk['text'], "text_pair": clean_answer} for chunk in context_list]

    # 2. top_k=None permet de récupérer les scores de TOUTES les classes (Entailment, Neutral, Contradiction)
    results = nli_judge(pairs, top_k=None)

    # 'results' est maintenant une liste de listes (une liste de scores par chunk)
    for chunk, chunk_scores in zip(context_list, results):

        # On cherche le score spécifique de la classe 'entailment'
        entailment_score = 0.0
        for score_dict in chunk_scores:
            if 'ENTAIL' in score_dict['label'].upper(): # Attrape 'ENTAILMENT' ou 'entailment'
                entailment_score = score_dict['score']
                break

        # On applique le seuil
        if entailment_score >= threshold:
            predicted_ids.append(chunk['id'])

    return predicted_ids

In [ ]:
PROMPT_NLI = """Role: You are a strict factual extraction agent.
Task: Answer the question using ONLY the provided context.

### CONSTRAINTS
1. THE SENTENCE MUST BE SELF-CONTAINED: Someone reading only your answer should understand the full fact without seeing the question. Write complete answers including any context needed to answer the question, based on the paragraphs.

### FORMAT:
<answer>[Your single, full, entity-rich sentence]</answer>

### CONTEXT:
{context_text}

### QUESTION:
{question}
"""

In [ ]:
results_nli = []

for index, row in ds_rag.head(20).iterrows():

    full_prompt = prepare_input(row, PROMPT_NLI)

    messages = [{"role": "user", "content": full_prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt"
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=150,
            do_sample=False
        )

    llm_answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    # --- NLI STEP ---
    pred_ids = scan_context_with_nli(llm_answer, row["contexts"])

    # --- METRICS ---
    gold_set = set(row["gold_ids"])
    pred_set = set(pred_ids)

    correct_ids = pred_set.intersection(gold_set)

    p = len(correct_ids) / len(pred_set) if pred_set else 0
    r = len(correct_ids) / len(gold_set) if gold_set else 0
    em = 1 if pred_set == gold_set else 0

    results_nli.append({"p": p, "r": r, "em": em})

    print(f"--- SAMPLE {index + 1} ---")
    print(f"Q: {row['question']}")
    print(f"A: {llm_answer}")
    print(f"Gold IDs: {row['gold_ids']}")
    print(f"NLI Pred IDs: {pred_ids}")
    print(f"Verdict: {'✅' if em else '❌'} (P: {p:.2f}, R: {r:.2f})")
    print("-" * 30)

    del inputs, outputs
    clean_memory()

# --- BILAN FINAL ---
total = len(results_nli)

print("\n" + "=" * 40)
print(f"BILAN FINAL NLI-SCAN ({total} SAMPLES)")
print("=" * 40)

print(f"Précision moyenne : {sum(m['p'] for m in results_nli) / total:.2%}")
print(f"Rappel moyen      : {sum(m['r'] for m in results_nli) / total:.2%}")
print(f"Exact Match Total : {sum(m['em'] for m in results_nli) / total:.2%}")

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 1 ---
Q: When was the institute that owned The Collegian founded?
A: The University of St. Thomas, which is affiliated with the institute that owns The Collegian, was founded in 1947, however, the institute that owns The Collegian, Houston Baptist University, was founded in 1960.
Gold IDs: [5, 9]
NLI Pred IDs: []
Verdict: ❌ (P: 0.00, R: 0.00)
------------------------------
--- SAMPLE 2 ---
Q: What year saw the creation of the region where the county of Hertfordshire is located?
A: <answer>The East of England region, where the county of Hertfordshire is located, was created in 1994.</answer>
Gold IDs: [6, 10]
NLI Pred IDs: []
Verdict: ❌ (P: 0.00, R: 0.00)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 3 ---
Q: When was the abolishment of the studio that distributed The Game?
A: <answer>PolyGram Filmed Entertainment was eventually sold to Seagram Company Ltd. in 1998 and was folded in 1999.</answer>
Gold IDs: [0, 17]
NLI Pred IDs: []
Verdict: ❌ (P: 0.00, R: 0.00)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

--- SAMPLE 13 ---
Q: What is the college Francis Walsingham attended an instance of?
A: <answer>Francis Walsingham matriculated at King's College, Cambridge, in 1548.</answer>
Gold IDs: [11, 13]
NLI Pred IDs: []
Verdict: ❌ (P: 0.00, R: 0.00)
------------------------------
--- SAMPLE 14 ---
Q: What type of university is the college Kyeon Mi-ri attended?
A: Kyeon Mi-ri attended a traditional arts high school and then a university, specifically Sejong University, which is a private university.
Gold IDs: [3, 15]
NLI Pred IDs: []
Verdict: ❌ (P: 0.00, R: 0.00)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 15 ---
Q: In what year was the author of The Insider's Guide to the Colleges established?
A: <answer>The Insider's Guide to the Colleges was established annually by the student editorial staff of the "Yale Daily News" for over four decades, with the first publication year not specified in the provided context.</answer>
Gold IDs: [17, 18]
NLI Pred IDs: []
Verdict: ❌ (P: 0.00, R: 0.00)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


KeyboardInterrupt: 

In [ ]:
# --- BILAN FINAL ---
total = len(results_nli)

print("\n" + "=" * 40)
print(f"BILAN FINAL NLI-SCAN ({total} SAMPLES)")
print("=" * 40)

print(f"Précision moyenne : {sum(m['p'] for m in results_nli) / total:.2%}")
print(f"Rappel moyen      : {sum(m['r'] for m in results_nli) / total:.2%}")
print(f"Exact Match Total : {sum(m['em'] for m in results_nli) / total:.2%}")

In [ ]:
import re
import torch
import gc
from transformers import pipeline


# ==========================================
# 1. INITIALISATION DU NLI JUDGE (BART)
# ==========================================
# On revient sur le classique BART MNLI
nli_judge = pipeline(
    "text-classification",
    model="facebook/bart-large-mnli",
    device=0
)

# ==========================================
# 2. PROMPTS
# ==========================================
PROMPT_NLI = """Role: You are a strict factual extraction agent.
Task: Answer the question using ONLY the provided context.

### CONSTRAINTS:
1. THE ANSWER MUST BE A SINGLE SENTENCE.
2. DO NOT use pronouns like "He", "It", or "They". Use the full names of entities.
3. THE SENTENCE MUST BE SELF-CONTAINED: Someone reading only your answer should understand the full fact without seeing the question.
4. NO EXTERNAL KNOWLEDGE: If the context doesn't have the answer, write "Information not available". You must rely ONLY on the context to answer.

### FORMAT:
<answer>[Your single, full, entity-rich sentence]</answer>

### CONTEXT:
{context_text}

### QUESTION:
{question}
"""

PROMPT_ATOMIC_FACTS = """Role: You are a strict linguistic expert.
Task: Break down the following complex sentence into simple, independent, atomic facts.
- Each fact must be a complete, self-contained sentence.
- Replace pronouns with the full entity names.
- Output ONLY the facts, one per line, separated by a bullet point (-).

Sentence: {complex_sentence}
"""

# ==========================================
# 3. FONCTION NLI ATOMIQUE
# ==========================================
def scan_context_with_nli_atomic(atomic_facts_list, context_list, threshold=0.6):
    """
    Vérifie si les chunks impliquent (entail) les faits atomiques.
    """
    predicted_ids = set()

    if not atomic_facts_list:
        return []

    for fact in atomic_facts_list:
        # Ordre standard d'évaluation NLI :
        # text = Premise (Le grand texte du chunk)
        # text_pair = Hypothesis (Le petit fait atomique qu'on veut prouver)
        pairs = [{"text": chunk['text'], "text_pair": fact} for chunk in context_list]

        # top_k=None assure qu'on récupère le score de 'entailment' même si 'neutral' est plus élevé
        results = nli_judge(pairs, top_k=None)

        for chunk, chunk_scores in zip(context_list, results):
            for score_dict in chunk_scores:
                # BART renvoie 'entailment', 'neutral', 'contradiction'
                if 'ENTAIL' in str(score_dict['label']).upper() and score_dict['score'] >= threshold:
                    predicted_ids.add(chunk['id'])
                    break # Ce chunk est validé pour ce fait, on passe au chunk suivant

    return list(predicted_ids)

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

In [ ]:
# ==========================================
# 4. BOUCLE PRINCIPALE
# ==========================================
results_nli = []

for index, row in ds_rag.head(20).iterrows():

    # --- ETAPE A : GENERATION DE LA REPONSE (Qwen) ---
    full_prompt = prepare_input(row, PROMPT_NLI)
    messages = [{"role": "user", "content": full_prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs.get("attention_mask"),
            max_new_tokens=150,
            do_sample=False
        )

    llm_answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    clean_llm_answer = re.sub(r'</?answer>', '', llm_answer).strip()

    del inputs, outputs
    clean_memory()

    # --- ETAPE B : DECOUPAGE EN FAITS ATOMIQUES (Qwen) ---
    split_prompt = PROMPT_ATOMIC_FACTS.format(complex_sentence=clean_llm_answer)
    messages_split = [{"role": "user", "content": split_prompt}]

    inputs_split = tokenizer.apply_chat_template(
        messages_split,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )
    inputs_split = {k: v.to(model.device) for k, v in inputs_split.items()}

    with torch.no_grad():
        outputs_split = model.generate(
            input_ids=inputs_split["input_ids"],
            attention_mask=inputs_split.get("attention_mask"),
            max_new_tokens=150,
            do_sample=False
        )

    facts_text = tokenizer.decode(
        outputs_split[0][inputs_split["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    atomic_facts = [fact.strip("- *").strip() for fact in facts_text.split('\n') if fact.strip()]

    del inputs_split, outputs_split
    clean_memory()

    # --- ETAPE C : EVALUATION NLI ---
    pred_ids = scan_context_with_nli_atomic(atomic_facts, row["contexts"])

    # --- METRICS ---
    gold_set = set(row["gold_ids"])
    pred_set = set(pred_ids)

    correct_ids = pred_set.intersection(gold_set)

    p = len(correct_ids) / len(pred_set) if pred_set else 0
    r = len(correct_ids) / len(gold_set) if gold_set else 0
    em = 1 if pred_set == gold_set else 0

    results_nli.append({"p": p, "r": r, "em": em})

    # --- AFFICHAGE ---
    print(f"--- SAMPLE {index + 1} ---")
    print(f"Q: {row['question']}")
    print(f"A (Brute): {llm_answer}")
    print("Faits Atomiques :")
    for f in atomic_facts:
        print(f"  - {f}")
    print(f"Gold IDs: {row['gold_ids']}")
    print(f"NLI Pred IDs: {pred_ids}")
    print(f"Verdict: {'✅' if em else '❌'} (P: {p:.2f}, R: {r:.2f})")
    print("-" * 30)

# ==========================================
# 5. BILAN FINAL
# ==========================================
total = len(results_nli)

print("\n" + "=" * 40)
print(f"BILAN FINAL NLI-SCAN ATOMIQUE BART ({total} SAMPLES)")
print("=" * 40)

print(f"Précision moyenne : {sum(m['p'] for m in results_nli) / total:.2%}")
print(f"Rappel moyen      : {sum(m['r'] for m in results_nli) / total:.2%}")
print(f"Exact Match Total : {sum(m['em'] for m in results_nli) / total:.2%}")

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 1 ---
Q: When was the institute that owned The Collegian founded?
A (Brute): The University of St. Thomas, which is affiliated with the institute that owns The Collegian, was founded in 1947, however, the institute that owns The Collegian, Houston Baptist University, was founded in 1960.
Faits Atomiques :
  - The University of St. Thomas was founded.
  - The University of St. Thomas is affiliated with an institute.
  - The institute that owns The Collegian owns The Collegian.
  - The institute that owns The Collegian was founded.
  - The institute that owns The Collegian is affiliated with Houston Baptist University.
  - Houston Baptist University owns The Collegian.
  - Houston Baptist University was founded in 1960.
  - The University of St. Thomas was founded in 1947.
Gold IDs: [5, 9]
NLI Pred IDs: [9]
Verdict: ❌ (P: 1.00, R: 0.50)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 2 ---
Q: What year saw the creation of the region where the county of Hertfordshire is located?
A (Brute): <answer>The East of England region, where the county of Hertfordshire is located, was created in 1994.</answer>
Faits Atomiques :
  - The East of England region exists.
  - The East of England region is located in England.
  - The county of Hertfordshire is located in the East of England region.
  - The county of Hertfordshire exists.
  - The year 1994 exists.
  - The East of England region was created in the year 1994.
Gold IDs: [6, 10]
NLI Pred IDs: [0, 1, 2, 3, 6, 8, 9, 10, 11, 12, 13, 14, 15, 17, 19]
Verdict: ❌ (P: 0.13, R: 1.00)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

--- SAMPLE 18 ---
Q: Where did the director who provided the lyrics to A Time for Singing attend school?
A (Brute): Gerald Freedman attended the Royal College of Art in London.
Faits Atomiques :
  - Gerald Freedman is a person.
  - Gerald Freedman attended an institution.
  - The institution is the Royal College of Art.
  - The Royal College of Art is located in London.
  - London is a city.
Gold IDs: [4, 13]
NLI Pred IDs: [8, 4]
Verdict: ❌ (P: 0.50, R: 0.50)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 19 ---
Q: When did the country formerly known as Zaire become independent?
A (Brute): Information not available.
Faits Atomiques :
  - The information is not available.
Gold IDs: [8, 14]
NLI Pred IDs: []
Verdict: ❌ (P: 0.00, R: 0.00)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 20 ---
Q: Where did Peter and Paul Fortress' designer die?
A (Brute): Information not available.
Faits Atomiques :
  - The information is not available.
Gold IDs: [3, 9]
NLI Pred IDs: []
Verdict: ❌ (P: 0.00, R: 0.00)
------------------------------

BILAN FINAL NLI-SCAN ATOMIQUE BART (20 SAMPLES)
Précision moyenne : 29.76%
Rappel moyen      : 42.50%
Exact Match Total : 10.00%


In [ ]:
# ==========================================
# 3. FONCTION NLI ATOMIQUE (CORRIGEE)
# ==========================================
def scan_context_with_nli_atomic(atomic_facts_list, context_list, threshold=0.5):
    """
    Vérifie chaque chunk contre chaque fait atomique.
    """
    predicted_ids = set() # Set pour éviter les doublons

    if not atomic_facts_list:
        return []

    for fact in atomic_facts_list:
        # ⚠️ LA CORRECTION EST ICI :
        # text = textA = Hypothesis (le fait atomique)
        # text_pair = textB = Premise (le chunk de texte complet)
        pairs = [{"text": fact, "text_pair": chunk['text']} for chunk in context_list]

        # top_k=None pour récupérer toutes les probabilités
        results = nli_judge(pairs, top_k=None)

        for chunk, chunk_scores in zip(context_list, results):
            for score_dict in chunk_scores:
                # On cherche l'entailment qui dépasse le seuil
                if 'ENTAIL' in str(score_dict['label']).upper() and score_dict['score'] >= threshold:
                    predicted_ids.add(chunk['id'])
                    break # Ce chunk valide ce fait, on passe au chunk suivant

    return list(predicted_ids)

In [ ]:
# ==========================================
# 4. BOUCLE PRINCIPALE
# ==========================================
results_nli = []

for index, row in ds_rag.head(20).iterrows():

    # --- ETAPE A : GENERATION DE LA REPONSE (Qwen) ---
    full_prompt = prepare_input(row, PROMPT_NLI)
    messages = [{"role": "user", "content": full_prompt}]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs.get("attention_mask"),
            max_new_tokens=150,
            do_sample=False
        )

    llm_answer = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    # Nettoyage des balises <answer>
    clean_llm_answer = re.sub(r'</?answer>', '', llm_answer).strip()

    del inputs, outputs
    clean_memory()

    # --- ETAPE B : DECOUPAGE EN FAITS ATOMIQUES (Qwen) ---
    split_prompt = PROMPT_ATOMIC_FACTS.format(complex_sentence=clean_llm_answer)
    messages_split = [{"role": "user", "content": split_prompt}]

    inputs_split = tokenizer.apply_chat_template(
        messages_split,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt"
    )
    inputs_split = {k: v.to(model.device) for k, v in inputs_split.items()}

    with torch.no_grad():
        outputs_split = model.generate(
            input_ids=inputs_split["input_ids"],
            attention_mask=inputs_split.get("attention_mask"),
            max_new_tokens=150,
            do_sample=False
        )

    facts_text = tokenizer.decode(
        outputs_split[0][inputs_split["input_ids"].shape[-1]:],
        skip_special_tokens=True
    ).strip()

    # Extraction des faits sous forme de liste
    atomic_facts = [fact.strip("- *").strip() for fact in facts_text.split('\n') if fact.strip()]

    del inputs_split, outputs_split
    clean_memory()

    # --- ETAPE C : EVALUATION NLI ---
    # On passe la liste des faits à notre fonction NLI (Attention : row["contexts"] avec le S)
    pred_ids = scan_context_with_nli_atomic(atomic_facts, row["contexts"])

    # --- METRICS ---
    gold_set = set(row["gold_ids"])
    pred_set = set(pred_ids)

    correct_ids = pred_set.intersection(gold_set)

    p = len(correct_ids) / len(pred_set) if pred_set else 0
    r = len(correct_ids) / len(gold_set) if gold_set else 0
    em = 1 if pred_set == gold_set else 0

    results_nli.append({"p": p, "r": r, "em": em})

    # --- AFFICHAGE ---
    print(f"--- SAMPLE {index + 1} ---")
    print(f"Q: {row['question']}")
    print(f"A (Brute): {llm_answer}")
    print("Faits Atomiques :")
    for f in atomic_facts:
        print(f"  - {f}")
    print(f"Gold IDs: {row['gold_ids']}")
    print(f"NLI Pred IDs: {pred_ids}")
    print(f"Verdict: {'✅' if em else '❌'} (P: {p:.2f}, R: {r:.2f})")
    print("-" * 30)

# ==========================================
# 5. BILAN FINAL
# ==========================================
total = len(results_nli)

print("\n" + "=" * 40)
print(f"BILAN FINAL NLI-SCAN ATOMIQUE ({total} SAMPLES)")
print("=" * 40)

print(f"Précision moyenne : {sum(m['p'] for m in results_nli) / total:.2%}")
print(f"Rappel moyen      : {sum(m['r'] for m in results_nli) / total:.2%}")
print(f"Exact Match Total : {sum(m['em'] for m in results_nli) / total:.2%}")

[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 1 ---
Q: When was the institute that owned The Collegian founded?
A (Brute): <answer>The University of St. Thomas, which is affiliated with the institute that owns The Collegian, was founded in 1947, however, the institute that owns The Collegian, Houston Baptist University, was founded in 1960.</answer>
Faits Atomiques :
  - The University of St. Thomas was founded.
  - The University of St. Thomas is affiliated with an institute.
  - The institute that owns The Collegian owns The Collegian.
  - The institute that owns The Collegian was founded.
  - The institute that owns The Collegian is affiliated with Houston Baptist University.
  - Houston Baptist University owns The Collegian.
  - Houston Baptist University was founded in 1960.
  - The University of St. Thomas was founded in 1947.
Gold IDs: [5, 9]
NLI Pred IDs: []
Verdict: ❌ (P: 0.00, R: 0.00)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 2 ---
Q: What year saw the creation of the region where the county of Hertfordshire is located?
A (Brute): <answer>The East of England region, where the county of Hertfordshire is located, was created in 1994.</answer>
Faits Atomiques :
  - The East of England region exists.
  - The East of England region is located in England.
  - The county of Hertfordshire is located in the East of England region.
  - The county of Hertfordshire exists.
  - The year 1994 exists.
  - The East of England region was created in the year 1994.
Gold IDs: [6, 10]
NLI Pred IDs: []
Verdict: ❌ (P: 0.00, R: 0.00)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


...

--- SAMPLE 7 ---
Q: When was the baseball team winning the world series in 2015 baseball created?
A (Brute): <answer>The Kansas City Royals, who won the 2015 World Series, were founded as an expansion franchise in 1969.</answer>
Faits Atomiques :
  - The Kansas City Royals were founded.
  - The Kansas City Royals were an expansion franchise.
  - The Kansas City Royals were founded in 1969.
  - The Kansas City Royals won the 2015 World Series.
  - The 2015 World Series was won by the Kansas City Royals.
Gold IDs: [5, 10]
NLI Pred IDs: []
Verdict: ❌ (P: 0.00, R: 0.00)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


--- SAMPLE 8 ---
Q: Where did the Baldevins bryllup director die?
A (Brute): <answer>There is no information about the death of George Schnéevoigt in the provided context.</answer>
Faits Atomiques :
  - There exists a statement about the death of George Schnéevoigt.
  - The statement is not present in the provided context.
  - The provided context is a set of information.
Gold IDs: [0, 10]
NLI Pred IDs: []
Verdict: ❌ (P: 0.00, R: 0.00)
------------------------------


[transformers] Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.


KeyboardInterrupt: 